In [13]:
from hpo_rl.main_scripts.plot import plot
from hpo_rl.main_scripts.check import check 
from hpo_rl.controller.controller import controller
from hpo_rl.models.simple_cnn import SimpleCNN
from hpo_rl.trainers.torch_trainer import TorchTrainer
from hpo_rl.data_processing.processors import pytorch_mnist_processor
import torch
import torch.optim as optim
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm


In [15]:
def run_experiment(config):
    parsed_config = check(config)
    # print(config, parsed_config, sep="\n\n", end="\n\n")
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    mode = parsed_config.get("mode")
    backend_name = config.get("backend").get("name")
    algorithm_name = config.get("algorithm").get("name")

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_dir = Path("logs") / algorithm_name / timestamp 
    log_dir.mkdir(parents=True, exist_ok=True)
    save_path = parsed_config.get("log_save_path", log_dir)

    expreiment_controller = controller(device=device, **parsed_config)
    if mode == "RL":
        expreiment_controller.train()
    best_result = expreiment_controller.inference()
    history = expreiment_controller.return_history()
    graphics = plot(history, best_result, save_path, expreiment_controller.backend)
    if backend_name == "function":
        graphics.plot_3d()
    graphics.plot_trajectory()

In [16]:
config = {
    "algorithm": {
        "name": "PPO", 
        "verbose": 1,
        "gamma": 0.95,
        "learning_rate": 0.001,
        "total_timesteps": 10000,
        "inference_timesteps": 50,
        "n_steps": 1000,
        "batch_size": 500,
        "policy": "MultiInputPolicy"
    },
    "env": {
        "name": "cycle_move_pipeline",
        "num_bins": 300,
        "max_steps": 20,
        "reward_mode": "per_step",
        "step_sizes": [5]
    },
    "backend": {
        "name": "function",
        "function": "rastrigin",
        "dimensions": 2
    }
}

In [17]:
run_experiment(config)

rastrigin: dims=2, bounds=(-5.12, 5.12), opt=0.000000
Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
1
ЖЖЖЖЖПАЖПажп 1
IDXs: 102 102
1
ЖЖЖЖЖПАЖПажп 1
IDXs: 29 29
0
IDXs: 102 97
1
ЖЖЖЖЖПАЖПажп 1
IDXs: 29 29
0
IDXs: 97 92
1
ЖЖЖЖЖПАЖПажп 1
IDXs: 29 29
0
IDXs: 92 87
0
IDXs: 29 24
0
IDXs: 87 82
1
ЖЖЖЖЖПАЖПажп 1
IDXs: 24 24
0
IDXs: 82 77
1
ЖЖЖЖЖПАЖПажп 1
IDXs: 24 24
2
IDXs: 77 82
1
ЖЖЖЖЖПАЖПажп 1
IDXs: 24 24
0
IDXs: 82 77
2
IDXs: 24 29
1
ЖЖЖЖЖПАЖПажп 1
IDXs: 77 77
1
ЖЖЖЖЖПАЖПажп 1
IDXs: 29 29
2
IDXs: 77 82
2
IDXs: 29 34
2
IDXs: 134 139
2
IDXs: 72 77
2
IDXs: 139 144
1
ЖЖЖЖЖПАЖПажп 1
IDXs: 77 77
2
IDXs: 144 149
0
IDXs: 77 72
1
ЖЖЖЖЖПАЖПажп 1
IDXs: 149 149
1
ЖЖЖЖЖПАЖПажп 1
IDXs: 72 72
2
IDXs: 149 154
0
IDXs: 72 67
0
IDXs: 154 149
2
IDXs: 67 72
0
IDXs: 149 144
2
IDXs: 72 77
0
IDXs: 144 139
0
IDXs: 77 72
0
IDXs: 139 134
0
IDXs: 72 67
2
IDXs: 134 139
1
ЖЖЖЖЖПАЖПажп 1
IDXs: 67 67
0
IDXs: 32 27
2
IDXs: 69 74
1
ЖЖЖЖЖПАЖПажп 1
IDXs: 27 27
0
IDXs: 74 69
0


KeyboardInterrupt: 

In [26]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=100, n_params=128):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, n_params) 
        self.fc2 = nn.Linear(n_params, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x))) 
        x = self.pool(F.relu(self.conv2(x))) 
        x = x.view(-1, 64 * 8 * 8) 
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [27]:
def objective_function(config, dict_config, num_epochs):
    param_values = {}
    for name in dict_config.keys():
        param_values[name] = config[name]

    n_params = param_values["n_params"]
    lr = param_values["lr"]
    batch_size = int(param_values["batch_size"])
    optimizer_name = param_values["optimizer"]

    transform = transforms.ToTensor()
    
    try:
        dataset = datasets.CIFAR100(root='./tmp_data', train=True, download=True, transform=transform)
    except:
        dataset = datasets.CIFAR100(root='./tmp_data', train=True, download=False, transform=transform)
        
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    model = SimpleCNN(num_classes=100, n_params=n_params) 
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.CrossEntropyLoss()
    if optimizer_name == "Adam":
        optimizer = optim.Adam(model.parameters(), lr=lr)
    else:
        optimizer = optim.SGD(model.parameters(), lr=lr)

    model.train()
    for epoch in tqdm(range(int(num_epochs))):
        for X, y in train_loader:
            X, y = X.to(device), y.to(device)
            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

    model.eval()
    val_loss, correct = 0.0, 0
    with torch.no_grad():
        for X, y in val_loader:
            X, y = X.to(device), y.to(device)
            outputs = model(X)
            loss = criterion(outputs, y)
            val_loss += loss.item()
            preds = outputs.argmax(dim=1)
            correct += (preds == y).sum().item()

    avg_val_loss = val_loss / len(val_loader)
    val_accuracy = correct / len(val_dataset)

    print(f"Config: {param_values}, ValLoss: {avg_val_loss:.4f}, ValAcc: {val_accuracy:.4f}")

    return avg_val_loss

In [29]:
config_objective = {
    "algorithm": {
        "name": "PPO", 
        "verbose": 1,
        "gamma": 0.95,
        "learning_rate": 0.001,
        "total_timesteps": 5,
        "inference_timesteps": 5,
        "n_steps": 5,
        "batch_size": 5,
        "policy": "MultiInputPolicy"
    },
    "env": {
        "name": "cycle_move_pipeline",
        "num_bins": 300,
        "max_steps": 6,
        "reward_mode": "per_step",
        "step_sizes": [1, 5, 25]
    },
    "backend": {
        "name": "objective",
        "num_epochs": 1,
        "objective_function": objective_function,
        "hp_space": {
            "lr": {
                "type": "float", 
                "min": 1e-6,
                "max": 1e-2
            },
            "batch_size": {
                "type": "categorical", 
                "values": [32, 64, 128]
            },
            "optimizer": {
                "type": "categorical", 
                "values": ["Adam", "SGD"]
            },
            "n_params": {
                "type": "categorical", 
                "values": [16, 32, 64] 
            }
        }
    }
    }
    

In [31]:
run_experiment(config_objective)

TypeError: check() takes 0 positional arguments but 1 was given

In [ ]:
config_TPE = {
    "backend": {
        "name": "function",
        "function": "rastrigin",
        "dimensions": 2
    },
    "algorithm": {
        "name": "TPE",
        "N_init": 10,
        "N_s": 10,
        "budget": 100,
        "separation_value": 0.2
    }
    }

In [ ]:
config_real = {
        "algorithm": {
        "name": "PPO", 
        "verbose": 1,
        "gamma": 0.95,
        "learning_rate": 0.001,
        "total_timesteps": 4,
        "inference_timesteps": 4,
        "n_steps": 2,
        "batch_size": 2,
        "policy": "MultiInputPolicy"
    },
    "env": {
        "name": "cycle_move_pipeline",
        "num_bins": 300,
        "max_steps": 2,
        "reward_mode": "per_step",
        "step_sizes": [1, 5, 25]
    },
    "backend": {
        "name": "real",
        "model": SimpleCNN,
        "trainer": TorchTrainer,
        "data_processor": pytorch_mnist_processor,
        "hp_space": {
            "n_params": {
                "refers_to": "model",
                "type": "int",
                "min": 1,
                "max": 512
            },
            "learning_rate":{
                "type": "float",
                "min": 0,
                "max": 0.1
            },
            "optimizer": {
                "refers_to": "train_loop",
                "type": "categorical",
                "values": ["SGD", "Adam"],
                "dependencies": ["learning_rate"]
            },
            "criterion": {
                "refers_to": "train_loop",
                "type": "categorical",
                "values": ["CrossEntropyLoss"]
            },
            "learning_rate": {
                "refers_to": "optimizer",
                "type": "float",
                "min": 0,
                "max": 1
            }
        }
    }
    }

In [ ]:
run_experiment(config_real)